In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("data.csv")

In [4]:
df = df.drop(columns=["id", "Unnamed: 32"])

X = df.drop("diagnosis", axis=1)

y = df["diagnosis"].map({
    "M": 1,
    "B": 0
})

print(X.shape)
print(y.value_counts())

(569, 30)
diagnosis
0    357
1    212
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
baseline = DecisionTreeClassifier(random_state=42)

baseline_cv = cross_val_score(
    baseline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Baseline CV Accuracy:", baseline_cv.mean())

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

baseline_test = accuracy_score(y_test, baseline_pred)

print("Baseline Test Accuracy:", baseline_test)

Baseline CV Accuracy: 0.9318681318681318
Baseline Test Accuracy: 0.9298245614035088


In [7]:
grid_params = {
    "max_depth": [3, 5, 7, None],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    grid_params,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("GridSearch CV Accuracy:", grid.best_score_)

Best Parameters: {'criterion': 'entropy', 'max_depth': 7, 'min_samples_split': 5}
GridSearch CV Accuracy: 0.945054945054945


In [8]:
grid_pred = grid.best_estimator_.predict(X_test)

grid_test = accuracy_score(y_test, grid_pred)

print("GridSearch Test Accuracy:", grid_test)

GridSearch Test Accuracy: 0.9473684210526315


In [9]:
random_params = {
    "max_depth": [2, 3, 4, 5, 6, 7, 8, None],
    "min_samples_split": [2, 4, 6, 8, 10],
    "min_samples_leaf": [1, 2, 4, 6],
    "criterion": ["gini", "entropy"]
}

random_search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    random_params,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
print("RandomizedSearch CV Accuracy:", random_search.best_score_)

Best Parameters: {'min_samples_split': 8, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'gini'}
RandomizedSearch CV Accuracy: 0.9384615384615385


In [10]:
random_pred = random_search.best_estimator_.predict(X_test)

random_test = accuracy_score(y_test, random_pred)

print("RandomizedSearch Test Accuracy:", random_test)

RandomizedSearch Test Accuracy: 0.9035087719298246


In [11]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_cv = cross_val_score(
    random_forest,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Random Forest CV Accuracy:", rf_cv.mean())

Random Forest CV Accuracy: 0.9626373626373628


In [12]:
random_forest.fit(X_train, y_train)

rf_pred = random_forest.predict(X_test)

rf_test = accuracy_score(y_test, rf_pred)

print("Random Forest Test Accuracy:", rf_test)

Random Forest Test Accuracy: 0.9736842105263158


In [13]:
results = pd.DataFrame({
    "Model": [
        "Baseline Decision Tree",
        "GridSearch Decision Tree",
        "RandomizedSearch Decision Tree",
        "Random Forest"
    ],

    "CV Accuracy": [
        baseline_cv.mean(),
        grid.best_score_,
        random_search.best_score_,
        rf_cv.mean()
    ],

    "Test Accuracy": [
        baseline_test,
        grid_test,
        random_test,
        rf_test
    ]
})

results

,Model,CV Accuracy,Test Accuracy
0,Baseline Decision Tree,0.931868,0.929825
1,GridSearch Decision Tree,0.945055,0.947368
2,RandomizedSearch Decision Tree,0.938462,0.903509
3,Random Forest,0.962637,0.973684


In [14]:
best_model = results.loc[
    results["Test Accuracy"].idxmax()
]

print("Best Model:", best_model["Model"])
print("Best Test Accuracy:", best_model["Test Accuracy"])

Best Model: Random Forest
Best Test Accuracy: 0.9736842105263158


# CONCLUSION

# 1. Decision Tree was used as the baseline classifier.
# 2. It was evaluated using 5-fold Cross Validation.
# 3. GridSearchCV was used to find the best parameter combination.
# 4. RandomizedSearchCV also searched for better parameters.
# 5. Random Forest was used as the ensemble model.
# 6. We compared all models using CV Accuracy and Test Accuracy.
# 7. The model with the highest test accuracy is selected as the best model.